# NanoScale-LM, compress and serve

A trained model is not a deployed model. This notebook takes a checkpoint and applies
the three levers that make it cheap to run, measuring each one:

| Lever | What it trades | Lossless? |
|---|---|---|
| **Distillation** | Capability for size, a smaller student trained on the teacher's distribution | No |
| **Quantization** | Precision for memory, 4-bit weights with per-group scales | No |
| **Speculative decoding** | Memory and compute for latency: a draft proposes, the target verifies | **Yes, exactly** |

The third is the one people find surprising, so it gets a proof rather than a claim.

Runs on CPU. A GPU makes it faster but changes no result.

## 0. Setup

In [ ]:
# Clone and install. ~90 seconds on a fresh Colab runtime.
!git clone --depth 1 https://github.com/vedant1711/nanoscale-lm 2>/dev/null || true
%cd nanoscale-lm
!pip install -q -e ".[data]"

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"torch {torch.__version__} | gpu: {gpu}")

In [ ]:
import copy

import torch

from nanoscale.config import TokenizerConfig, load_experiment
from nanoscale.data.toy import generate_corpus
from nanoscale.eval import perplexity
from nanoscale.tokenizer import BPETokenizer
from nanoscale.train import TokenBatcher, Trainer, build_packed_tokens

# Train a small teacher so the notebook is self-contained (~2 min on CPU).
tok = BPETokenizer.train(generate_corpus(seed=1337), TokenizerConfig(vocab_size=1024))
cfg = load_experiment(tier="nano", overrides=["train.device=cpu"])
trainer = Trainer(cfg, tokenizer=tok, out_dir="runs/nb/teacher")
trainer.train()
teacher = trainer.model

data = build_packed_tokens(cfg.data, tok)
eval_batches = TokenBatcher(data.val, seq_len=cfg.data.seq_len, batch_size=8, shuffle=False).take(
    16
)

base = perplexity(teacher, eval_batches)
print(f"teacher perplexity {base.perplexity:.4f}")

## 1. Distillation

Three objectives, implemented in `src/nanoscale/distill/`:

* **Forward KL** (Hinton et al.); `KL(teacher ‖ student)` on softened logits, scaled by
  `τ²` so the gradient magnitude does not depend on the temperature. Mode-*covering*: the
  student must put mass everywhere the teacher does, which for a much smaller student
  means smearing probability over things it cannot represent.
* **SeqKD** (Kim & Rush): just fine-tune on the teacher's greedy outputs. Crude, cheap,
  and often competitive.
* **On-policy reverse KL** (MiniLLM), `KL(student ‖ teacher)`, optimized as a policy
  gradient on the student's *own* samples. Mode-*seeking*: the student is allowed to
  ignore parts of the teacher's distribution, which is the right trade at 17× compression.

Reverse KL needs an MLE warm-start. Without it the policy gradient optimizes a student
that generates noise, and the run never recovers, the first version of this experiment
produced a perplexity of 1028 for exactly that reason.

In [ ]:
from nanoscale.config import draft_model_config
from nanoscale.distill import DistillTrainer
from nanoscale.model import build_model

# Same recipe as scripts/distill_compare.py: 600 steps of which the first 300 are a
# plain-MLE warm-start applied identically to every objective, lr 1e-3, seq_len 128.
train_batcher = TokenBatcher(data.train, seq_len=128, batch_size=8, seed=1337)

students = {}
for method in ["forward_kl", "seqkd", "reverse_kl"]:
    dcfg = cfg.distill.merged(
        method=method, max_steps=600, warmup_steps=300, lr=1e-3, seq_len=128, device="cpu"
    )
    dt = DistillTrainer(
        teacher,
        build_model(draft_model_config(cfg.model)),
        tok,
        dcfg,
        train_batcher=train_batcher,
        val_batches=eval_batches,
        out_dir=f"runs/nb/distill/{method}",
    )
    r = dt.train()
    students[method] = dt.student
    ppl = perplexity(dt.student, eval_batches).perplexity
    print(f"{method:>12}: perplexity {ppl:>8.4f}   ({r.student_params:,} params)")

print(
    f"\ncompression: {r.teacher_params:,} -> {r.student_params:,} "
    f"({r.teacher_params / r.student_params:.1f}x)"
)

## 2. Quantization

`RTN` rounds each weight to the nearest grid point independently. `GPTQ` does better by
noticing that the weights are not independent: it uses the Hessian of the layer's
reconstruction error, `H = 2XXᵀ`, to push the error of each quantized column into the
columns that have not been quantized yet.

At 4 bits on a 5M-parameter model the two are nearly tied: worth reporting as a null
result. The gap opens at 2 bits, where independent rounding falls apart.

In [ ]:
from nanoscale.quantize import GPTQQuantizer, quantize_rtn

calib = TokenBatcher(data.train, seq_len=cfg.data.seq_len, batch_size=8, seed=1337).take(8)

rows, gptq4 = [], None
for bits in [8, 4, 3, 2]:
    m_rtn = copy.deepcopy(teacher)
    quantize_rtn(m_rtn, bits=bits, group_size=64)

    m_gptq = copy.deepcopy(teacher)
    q = GPTQQuantizer(m_gptq, bits=bits, group_size=64, act_order=True)
    q.collect([b.inputs for b in calib])
    q.apply()

    if bits == 4:
        gptq4 = m_gptq
    rows.append(
        (
            bits,
            perplexity(m_rtn, eval_batches).perplexity,
            perplexity(m_gptq, eval_batches).perplexity,
        )
    )

print(f"{'bits':>5} {'RTN':>10} {'GPTQ':>10}")
print(f"{'fp32':>5} {base.perplexity:>10.4f} {base.perplexity:>10.4f}")
for bits, r, g in rows:
    print(f"{bits:>5} {r:>10.4f} {g:>10.4f}")

## 3. Speculative decoding

A small draft model proposes `γ` tokens; the target scores all `γ+1` positions in **one**
forward pass; each proposal is accepted with probability `min(1, p(x)/q(x))`, and the
first rejection is replaced by a sample from the residual `norm(max(0, p − q))`.

That accept/reject rule is not an approximation. The output distribution is *exactly*
the target's; the draft only changes how many target forward passes it takes to get
there. The cell below checks that empirically instead of asking you to believe it.

In [ ]:
from nanoscale.specdec import SpeculativeSampler, autoregressive_baseline

prompt = torch.tensor([tok.encode("It was a sunny day. Lily went to", add_bos=True)])

drafts = {
    "untrained": build_model(draft_model_config(cfg.model)),  # worst case
    "distilled": students["reverse_kl"],  # what you would ship
}

print(f"{'draft':>10} {'gamma':>6} {'accept rate':>12} {'tokens/target pass':>20}")
for name, d in drafts.items():
    for gamma in [1, 2, 4, 6, 8]:
        r = SpeculativeSampler(teacher, d, gamma=gamma).generate(
            prompt, max_new_tokens=64, generator=torch.Generator().manual_seed(0)
        )
        print(f"{name:>10} {gamma:>6} {r.acceptance_rate:>12.3f} {r.mean_accepted_length:>20.2f}")

draft = drafts["distilled"]

In [ ]:
from collections import Counter

# Losslessness, checked two ways.

# (a) Greedy decoding must match exactly, token for token.
a = SpeculativeSampler(teacher, draft, gamma=4, temperature=0.0).generate(prompt, max_new_tokens=32)
b = autoregressive_baseline(teacher, prompt, max_new_tokens=32, temperature=0.0)
assert torch.equal(a.tokens, b.tokens)
print("greedy: identical\n")

# (b) The *sampled* distributions must agree. This model is very confident, so we raise
# the temperature to get a distribution with enough entropy for the comparison to mean
# anything -- at T=1 it puts ~all its mass on one token and any sampler would "match".
T, N = 1.5, 600
spec, base_ = Counter(), Counter()
for i in range(N):
    spec[
        int(
            SpeculativeSampler(teacher, draft, gamma=4, temperature=T)
            .generate(prompt, max_new_tokens=1, generator=torch.Generator().manual_seed(i))
            .tokens[0, -1]
        )
    ] += 1
    base_[
        int(
            autoregressive_baseline(
                teacher,
                prompt,
                max_new_tokens=1,
                temperature=T,
                generator=torch.Generator().manual_seed(i),
            ).tokens[0, -1]
        )
    ] += 1

tv = 0.5 * sum(abs(spec[k] - base_[k]) for k in set(spec) | set(base_)) / N
print(f"{'token':>16} {'speculative':>12} {'target':>8}")
for tid, _ in base_.most_common(6):
    print(f"{tok.decode([tid])!r:>16} {spec[tid]:>12} {base_[tid]:>8}")
print(
    f"\ntotal variation distance: {tv:.3f}  "
    f"(sampling noise alone is ~{(len(set(base_)) / N) ** 0.5:.3f} at N={N})"
)

## 4. Compose the levers, and measure honestly

Quantization and speculation are orthogonal; you can quantize the target *and* speculate
against it. Distillation is not orthogonal to either; the distilled student **is** the
draft model.

In [ ]:
from nanoscale.bench import model_memory_bytes
from nanoscale.quantize import effective_bits

MB = 2**20
variants = [
    ("base (fp32)", teacher, None),
    ("distilled (reverse-KL)", students["reverse_kl"], None),
    ("GPTQ 4-bit", gptq4, effective_bits(4, group_size=64)),
]

print(f"{'variant':>24} {'weights':>10} {'perplexity':>11}")
for name, m, bits in variants:
    mb = model_memory_bytes(m, weight_bits=bits) / MB
    print(f"{name:>24} {mb:>7.2f} MB {perplexity(m, eval_batches).perplexity:>11.4f}")

# The full harness -- latency percentiles, KV footprint, acceptance rates, every
# variant including the composed ones -- is scripts/bench_all.py.

### Read the numbers with the caveat attached

On this hardware, at this scale, **speculative decoding is slower in wall-clock than
plain autoregressive decoding**, and 4-bit quantization is not faster than fp32. Both
results are real and both are reported in `docs/results.md`.

The reason is that neither method's cost model applies at 5M parameters on a CPU. Both
exist to relieve *memory bandwidth*: a decode step at 7B parameters is bounded by the
time to stream weights from HBM, so halving the bytes or amortizing a pass over several
tokens wins. At 5M parameters a forward pass is dominated by Python dispatch overhead,
which speculation *adds* to and quantization does nothing about.

What does transfer, because it is hardware-independent:

* the **target forward passes saved** (measured: 2.94 tokens per target pass at γ=6),
* the **weight footprint** at a given bit-width,
* the **exactness** of the speculative output distribution,
* the **rank ordering** of the distillation objectives.

That is the honest boundary of what a 5M-parameter CPU experiment can tell you, and
saying so is more useful than a speedup chart that would not survive contact with a real
deployment.